In [2]:
import pandas as pd
df = pd.read_csv("runs.csv")
g = df[df["model_class"]=="GRUStacked"].groupby(["input_width","learning_rate"])["val_loss_last"]
print(g.agg(['mean','std','count']).sort_values('mean').head(10))

# feature ablation effect (close vs close+open), matched by other params:
base = df[(df.model_class=="GRUStacked") & (df.features=="close")]
plus = df[(df.model_class=="GRUStacked") & (df.features=="close,open")]
m = base.merge(plus, on=["input_width","label_width","batch_size","learning_rate","num_epochs","windows_normalization_length"], suffixes=("_base","_plus"))
m["delta"] = m["val_loss_last_plus"] - m["val_loss_last_base"]
m["delta"].describe()  # <0 means adding "open" helped


                               mean       std  count
input_width learning_rate                           
200         0.0010         0.001759  0.000409     28
100         0.0005         0.001776  0.000365     36
            0.0010         0.001777  0.000352     36
200         0.0005         0.001786  0.000389     26


count    48.000000
mean     -0.000147
std       0.000098
min      -0.000364
25%      -0.000196
50%      -0.000134
75%      -0.000086
max       0.000029
Name: delta, dtype: float64

In [3]:
!pip install statsmodels 
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import wilcoxon
import statsmodels.api as sm
import statsmodels.formula.api as smf

CSV = "runs.csv"  # <-- your file path
df = pd.read_csv(CSV)

# Clean columns (sometimes strings/numbers mixed)
for c in ["input_width","label_width","batch_size","num_epochs","num_features","windows_normalization_length"]:
    if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
if "learning_rate" in df: df["learning_rate"] = pd.to_numeric(df["learning_rate"], errors="coerce")

# Only finished runs, and with val_loss_last present
df = df[(df["Status"].str.upper() == "FINISHED") & df["val_loss_last"].notna()]

def paired_feature_test(df_model, base="close", variant="close,open"):
    """Match runs by hyperparams; compute delta = variant - base; Wilcoxon paired test."""
    keys = ["input_width","label_width","batch_size","learning_rate","num_epochs","windows_normalization_length"]
    # Some exports have a 'coin' column; keep it in the match if present
    if "coin" in df_model.columns: keys = ["coin"] + keys

    base_df   = df_model[df_model["features"]==base][keys + ["val_loss_last"]].copy()
    variant_df= df_model[df_model["features"]==variant][keys + ["val_loss_last"]].copy()

    m = pd.merge(base_df, variant_df, on=keys, suffixes=("_base","_plus"))
    if m.empty:
        return {"n_pairs": 0}

    m["delta"] = m["val_loss_last_plus"] - m["val_loss_last_base"]  # <0 means variant helped
    deltas = m["delta"].values

    res = {"n_pairs": int(len(deltas)),
           "mean_delta": float(np.mean(deltas)),
           "std_delta": float(np.std(deltas, ddof=1)) if len(deltas) > 1 else 0.0,
           "median_delta": float(np.median(deltas)),
           "min_delta": float(np.min(deltas)),
           "max_delta": float(np.max(deltas))}
    # Effect size (paired Cohen's d = mean / std)
    res["cohens_d_paired"] = float(res["mean_delta"] / res["std_delta"]) if res["std_delta"] > 0 else np.nan

    # Wilcoxon signed-rank (paired, non-parametric)
    try:
        stat, p = wilcoxon(deltas, zero_method="wilcox", correction=True, alternative="two-sided")
        res["wilcoxon_stat"] = float(stat)
        res["wilcoxon_p"] = float(p)
    except ValueError:
        res["wilcoxon_stat"] = np.nan
        res["wilcoxon_p"] = np.nan

    return res

def two_way_anova(df_model, feature_choice="close,open"):
    """ANOVA for input_width × learning_rate on val_loss_last, using a single feature set."""
    sub = df_model[df_model["features"]==feature_choice].copy()
    sub = sub.dropna(subset=["input_width","learning_rate","val_loss_last"])
    # need at least 2 levels in each factor and enough rows
    if sub["input_width"].nunique() < 2 or sub["learning_rate"].nunique() < 2 or len(sub) < 8:
        return {"note": "not enough data for 2-way ANOVA"}

    # Treat both as categorical (levels)
    sub["input_width"] = sub["input_width"].astype("category")
    sub["learning_rate"] = sub["learning_rate"].astype("category")

    model = smf.ols("val_loss_last ~ C(input_width) * C(learning_rate)", data=sub).fit()
    anova_tbl = sm.stats.anova_lm(model, typ=2)
    # η² effect size per term = SS_term / SS_total
    ss_total = float(((sub["val_loss_last"] - sub["val_loss_last"].mean())**2).sum())
    eff = {}
    for term in anova_tbl.index:
        ss = float(anova_tbl.loc[term, "sum_sq"])
        eff[f"eta2_{term}"] = (ss / ss_total) if ss_total > 0 else np.nan

    # pack small summary
    out = {
        "anova": {
            "C(input_width)_p": float(anova_tbl.loc["C(input_width)","PR(>F)"]) if "C(input_width)" in anova_tbl.index else np.nan,
            "C(learning_rate)_p": float(anova_tbl.loc["C(learning_rate)","PR(>F)"]) if "C(learning_rate)" in anova_tbl.index else np.nan,
            "interaction_p": float(anova_tbl.loc["C(input_width):C(learning_rate)","PR(>F)"]) if "C(input_width):C(learning_rate)" in anova_tbl.index else np.nan,
        },
        "eta2": eff,
        "n_rows": int(len(sub)),
        "levels": {
            "input_width": sorted(sub["input_width"].unique()),
            "learning_rate": sorted(sub["learning_rate"].unique()),
        }
    }
    return out

results = {}
for model_name in sorted(df["model_class"].dropna().unique()):
    dfm = df[df["model_class"]==model_name].copy()
    abl = paired_feature_test(dfm, base="close", variant="close,open")
    an = two_way_anova(dfm, feature_choice="close,open")
    results[model_name] = {"feature_ablation_close_vs_close_open": abl,
                           "anova_inputwidth_x_lr": an}

# Pretty print
import json
print(json.dumps(results, indent=2, default=str))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 24.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.9/232.9 kB 28.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
{
  "ConvFC": {
    "feature_ablation_close_vs_close_open": {
      "n_pairs": 48,
      "mean_delta": -0.00015579650827428958,
      "std_delta": 0.0001276572633190714,
      "median_delta": -0.00016600338818049975,
      "min_delta": -0.0005185126757490002,
      "max_delta": 0.0002057253109144998,
      "cohens_d_paired": -1.2204280761125663,
      "wilcoxon_stat": 47.0,
      "wilcoxon_p": 1.545501504551794e-10
    },
    "anova_inputwidth_x_lr": {
      "anova": {
        "C(input_width)_p": 0.5681727299451791,
        "C(learning_rate)_p": 0.8682116703526026,
        "interaction_p": 0.9865400336965664
      },
      "eta2": {
        "eta2_C(input_width)": 0.007455247784867661,
        "eta2_C(learning_rat